Dynamic_Topic_Analysis_Platform.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1HnQLjNYeCvMRRS-_9lzDiihjuCRTXbsW

# 📰 Dynamic Topic Analysis Platform
### Dataset: **Full 20 Newsgroups** (~17,886 docs) | **30 Groups** | **Bigrams** | **Flan-T5 Auto Topic Labeling**
> **Fully Automatic Topic Labels:** Uses `google/flan-t5-base` text generation — feeds each topic's top words as a prompt and the model generates a descriptive label on its own. Zero manual input, zero candidate labels, zero hardcoded keywords.
>
> **Pipeline:** Load 20NG → Preprocess → Bigrams → Sub-topic Split (20→30) → LDA/NMF (30 topics) → **Flan-T5 Label Generation** → Sentiment → Summarization → Dashboard

---

## ⚙️ Step 1 — Install & Import Dependencies

In [ ]:
!pip install -q --upgrade transformers
!pip install -q gensim pyLDAvis nltk textblob wordcloud matplotlib seaborn pandas scikit-learn torch sentencepiece

import warnings
warnings.simplefilter("ignore", DeprecationWarning)

import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob

import gensim
from gensim import corpora
from gensim.models import LdaModel
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.datasets import fetch_20newsgroups

from wordcloud import WordCloud
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print('✅ All dependencies loaded!')

---
## 📥 Step 2 — Load 20 Newsgroups Dataset

In [ ]:
# Load FULL 20 Newsgroups dataset — train + test, no sampling
CATEGORIES = None  # all 20 categories

newsgroups_train = fetch_20newsgroups(
    subset='train',
    categories=CATEGORIES,
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)
newsgroups_test = fetch_20newsgroups(
    subset='test',
    categories=CATEGORIES,
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)

# Combine train + test — full corpus, no sampling
all_texts   = newsgroups_train.data + newsgroups_test.data
all_targets = list(newsgroups_train.target) + list(newsgroups_test.target)
target_names = newsgroups_train.target_names

df = pd.DataFrame({'text': all_texts, 'category_id': all_targets})
df['group'] = df['category_id'].map(dict(enumerate(target_names)))
df = df[df['text'].str.strip().str.len() > 50].reset_index(drop=True)

print(f'✅ Full 20 Newsgroups dataset loaded (train + test, no sampling)!')
print(f'   Total documents  : {len(df):,}')
print(f'   Categories       : {df["group"].nunique()}')
print(f'\nDocument count per category:')
print(df['group'].value_counts().to_string())

---
## 🧹 Step 3 — Data Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
extra_stops = {
    # common filler words
    'would','could','said','one','get','like','know',
    'think','use','make','say','see','may','well',
    'much','many','way','good','time','people','thing',
    # conversational noise common in newsgroups
    'dont','didnt','youre','cant','isnt','wasnt','wont',
    'going','really','something','anything','nothing','everything',
    'someone','anyone','everyone','want','back','take',
    'right','left','first','still','might','come','look','tell',
    'thank','thanks','please','help','need','find','work','used',
    'just','also','even','well','got','went','came','told','read','mean',
    'write','wrote','post','email','mail','list','group','article',
    'idk','thats','whats','heres','doesnt','havent',
}
stop_words.update(extra_stops)

def preprocess(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|\S+@\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in stop_words and len(t) > 2]
    return tokens

print('Preprocessing documents...')
df['tokens'] = df['text'].apply(preprocess)
df['clean_text'] = df['tokens'].apply(lambda t: ' '.join(t))
df = df[df['tokens'].map(len) > 5].reset_index(drop=True)

print(f'Preprocessing done!')
print(f'   Docs after filtering : {len(df)}')
print(f'   Total tokens         : {df["tokens"].map(len).sum():,}')
print(f'\nSample tokens (doc 0): {df["tokens"][0][:12]}')

---
## 🔗 Step 3A — Bigram Detection
> Bigrams capture compound terms like `space_shuttle`, `gun_control`, `hard_drive` that lose meaning when split into single words. We use `gensim.models.Phrases` with a min_count and threshold to filter only meaningful pairs.

In [ ]:
from gensim.models import Phrases
from gensim.models.phrases import Phraser
# re already imported above; aliased here for clarity
_re = re

token_list = df['tokens'].tolist()

print('⏳ Training bigram model...')
bigram_model = Phrases(token_list, min_count=10, threshold=15)
bigram_phraser = Phraser(bigram_model)

print('⏳ Training trigram model...')
trigram_model = Phrases(bigram_phraser[token_list], min_count=5, threshold=15)
trigram_phraser = Phraser(trigram_model)

def is_clean_token(t):
    # Remove repeating-pattern noise tokens like maxaxaxaxax
    if _re.search(r'(..+)\1{3,}', t): return False
    if len(t) > 40: return False
    return True

df['tokens_bigram'] = [
    [t for t in trigram_phraser[bigram_phraser[doc]] if is_clean_token(t)]
    for doc in token_list
]
df['clean_text_bigram'] = df['tokens_bigram'].apply(lambda t: ' '.join(t))

all_ngrams = [t for doc in df['tokens_bigram'] for t in doc if '_' in t]
from collections import Counter
top_ngrams = Counter(all_ngrams).most_common(30)
print(f'\n✅ Bigram/Trigram detection complete!')
print(f'   Unique n-grams  : {len(set(all_ngrams)):,}')
print(f'   Total occurrences: {len(all_ngrams):,}')
print(f'\nTop 30 bigrams/trigrams:')
for ngram, count in top_ngrams:
    print(f'   {ngram.replace("_", " "):35s} -> {count:,}')
print(f'\nDoc 0 BEFORE: {df["tokens"][0][:8]}')
print(f'Doc 0 AFTER : {df["tokens_bigram"][0][:8]}')

---
## ✂️ Step 3B — Sub-topic Splitting (20 → 30 Groups)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Sub-topic splitting: expand 10 large categories into 2 each
# giving us 10 original + (10×2) = 30 groups total
# ─────────────────────────────────────────────────────────────

SUBTOPIC_RULES = {
    'sci.space': {
        'sci.space.nasa':      ['nasa', 'shuttle', 'launch', 'mission', 'astronaut', 'rocket'],
        'sci.space.astronomy': ['planet', 'star', 'galaxy', 'telescope', 'orbit', 'comet', 'solar']
    },
    'talk.politics.guns': {
        'politics.guns.control': ['ban', 'control', 'regulation', 'law', 'crime', 'restrict'],
        'politics.guns.rights':  ['right', 'amendment', 'constitution', 'freedom', 'defend', 'arm']
    },
    'talk.politics.mideast': {
        'politics.mideast.israel': ['israel', 'israeli', 'jewish', 'jews', 'jerusalem', 'zion'],
        'politics.mideast.arab':   ['arab', 'palestinian', 'muslim', 'islam', 'iran', 'iraq', 'war']
    },
    'comp.graphics': {
        'comp.graphics.software': ['software', 'program', 'render', 'opengl', 'format', 'image'],
        'comp.graphics.hardware': ['card', 'monitor', 'display', 'screen', 'video', 'resolution']
    },
    'rec.sport.hockey': {
        'sport.hockey.nhl':     ['nhl', 'league', 'team', 'player', 'season', 'coach', 'game'],
        'sport.hockey.general': ['ice', 'skate', 'stick', 'goal', 'puck', 'rink', 'score']
    },
    'sci.med': {
        'sci.med.disease':    ['disease', 'cancer', 'virus', 'infection', 'patient', 'symptom'],
        'sci.med.treatment':  ['drug', 'treatment', 'doctor', 'hospital', 'medicine', 'therapy']
    },
    'comp.sys.ibm.pc.hardware': {
        'comp.hardware.cpu':    ['cpu', 'processor', 'intel', 'speed', 'memory', 'ram', 'chip'],
        'comp.hardware.drives': ['drive', 'disk', 'hard', 'storage', 'ide', 'scsi', 'bios']
    },
    'soc.religion.christian': {
        'religion.christian.bible':  ['bible', 'scripture', 'verse', 'testament', 'jesus', 'christ'],
        'religion.christian.church': ['church', 'faith', 'prayer', 'worship', 'god', 'sin', 'soul']
    },
    'rec.autos': {
        'rec.autos.buying':  ['buy', 'price', 'cost', 'dealer', 'model', 'year', 'sale'],
        'rec.autos.repair':  ['engine', 'repair', 'oil', 'brake', 'tire', 'mechanic', 'fix']
    },
    'sci.electronics': {
        'sci.electronics.circuit': ['circuit', 'voltage', 'resistor', 'capacitor', 'amp', 'signal'],
        'sci.electronics.devices': ['device', 'chip', 'board', 'arduino', 'sensor', 'power', 'battery']
    }
}

def assign_subtopic(row):
    cat = row['group']
    if cat not in SUBTOPIC_RULES:
        return cat  # keep original label for the 10 unsplit categories
    text_lower = row['text'].lower()
    scores = {}
    for subtopic, keywords in SUBTOPIC_RULES[cat].items():
        scores[subtopic] = sum(text_lower.count(kw) for kw in keywords)
    # If no keywords match, assign first subtopic as default
    if max(scores.values()) == 0:
        return list(SUBTOPIC_RULES[cat].keys())[0]
    return max(scores, key=scores.get)

print('⏳ Assigning sub-topics...')
df['group'] = df.apply(assign_subtopic, axis=1)

print(f'✅ Sub-topic splitting complete!')
print(f'   Original categories : 20')
print(f'   Final groups        : {df["group"].nunique()}')
print(f'\nDocument count per group:')
print(df['group'].value_counts().to_string())

---
## 🗂️ Step 4 — Topic Modeling
### 4A — LDA (Latent Dirichlet Allocation) with 30 Topics
> **Note:** LDA is unsupervised — it finds statistical patterns in text independently of our rule-based 30 sub-groups. The 30 sub-groups label documents for evaluation, while LDA discovers latent themes. We use `NUM_TOPICS=30` to match the group count, but topic-group alignment is not guaranteed by design.
> Step 4B uses NMF as a comparison — both methods are evaluated side by side.

In [ ]:
NUM_TOPICS = 30

dictionary = corpora.Dictionary(df['tokens_bigram'].tolist())
dictionary.filter_extremes(no_below=3, no_above=0.7)
corpus = [dictionary.doc2bow(tok) for tok in df['tokens_bigram'].tolist()]

print(f'Training LDA with {NUM_TOPICS} topics...')
lda_model = LdaModel(
    corpus=corpus, id2word=dictionary, num_topics=NUM_TOPICS,
    random_state=42, passes=10, alpha='auto', eta='auto'
)
print('LDA trained!')

from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch
from IPython.display import display

print('Loading Flan-T5...')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
t5_tokenizer = T5Tokenizer.from_pretrained('google/flan-t5-base')
t5_model = T5ForConditionalGeneration.from_pretrained('google/flan-t5-base').to(device)
t5_model.eval()
print(f'Flan-T5 loaded on {device}!')

def generate_topic_label(top_words):
    noise = {
        'dont','want','really','something','going','take','right',
        'back','still','might','anyone','thanks','please','need',
        'help','find','work','used','just','tell','read','mean',
        'idk','thats','doesnt','cant','youre','didnt','anything',
        'sure','little','better','never','probably','actually',
        'things','thing','also','even','good','best','great',
        'maybe','echo','support','results','software','convert',
        'display','speed','application','fonts','controller'
    }
    clean = [w.replace('_',' ') for w in top_words[:10]
             if w.lower().replace('_','') not in noise][:5]
    if not clean:
        clean = [w.replace('_',' ') for w in top_words[:3]]
    kw = ', '.join(clean)
    prompt = (
        'Give a clear, specific 2-4 word noun phrase topic label (no verbs, no articles).\n'
        'Keywords: space, shuttle, nasa, orbit -> Label: NASA space missions\n'
        'Keywords: israel, arab, jewish, state -> Label: Israeli Arab conflict\n'
        'Keywords: chip, encryption, key, security -> Label: encryption security\n'
        'Keywords: drive, scsi, disk, controller -> Label: SCSI disk drives\n'
        'Keywords: christian, bible, jesus, church -> Label: Christian Bible study\n'
        'Keywords: drug, patient, doctor, disease -> Label: medical treatment\n'
        f'Keywords: {kw} -> Label:'
    )
    inputs = t5_tokenizer(
        prompt, return_tensors='pt', truncation=True, max_length=150
    ).to(device)
    with torch.no_grad():
        outputs = t5_model.generate(
            **inputs, max_new_tokens=8, num_beams=4,
            early_stopping=True, do_sample=False
        )
    label = t5_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    # Strip all prompt echo artifacts
    label = re.sub(r'->\s*Label:?\s*', '', label).strip()
    label = re.sub(r'^(Label:|Topic:)?\s*', '', label, flags=re.IGNORECASE).strip()
    label = re.sub(r'\s*->.*$', '', label).strip()  # remove trailing arrow and anything after
    # Remove stray commas and normalize spaces
    label = re.sub(r'\s*,\s*', ' ', label).strip(' ,;:-')
    label = re.sub(r'\s+', ' ', label).strip()
    # Trim to max 6 words
    label = ' '.join(label.split()[:6])
    # Fallback if single word or empty
    if len(label.split()) <= 1:
        label = ' '.join(clean[:2])
    return label

print(f'Generating labels for {NUM_TOPICS} topics...')
topic_labels = {}
table_rows = []
for i, topic in lda_model.print_topics(num_topics=NUM_TOPICS, num_words=10):
    words = [w.split('*')[1].replace('"','').strip() for w in topic.split('+')]
    label = generate_topic_label(words)
    topic_labels[i] = label
    table_rows.append({
        'Topic No.': f'Topic {i+1}',
        'Generated Label': label,
        'Top Keywords': ', '.join(words[:7])
    })
    print(f'  Topic {i+1:2d} -> {label}')

df_topics = pd.DataFrame(table_rows)
styled = (
    df_topics.style
    .set_properties(**{
        'text-align': 'left',
        'font-size': '13px',
        'padding': '6px 14px',
        'color': '#000000',
        'background-color': '#ffffff'
    })
    .set_table_styles([{
        'selector': 'th',
        'props': [('background-color','#2c3e50'),('color','#ffffff'),
                  ('font-size','13px'),('padding','8px 14px'),('text-align','left')]
    },{
        'selector': 'tr:nth-child(even)',
        'props': [('background-color','#f0f4f8'),('color','#000000')]
    },{
        'selector': 'tr:nth-child(odd)',
        'props': [('background-color','#ffffff'),('color','#000000')]
    },{
        'selector': 'td',
        'props': [('color','#000000')]
    },{
        'selector': 'caption',
        'props': [('font-size','15px'),('font-weight','bold'),
                  ('padding','12px'),('color','#2c3e50'),('text-align','left')]
    }])
    .set_caption('LDA Topic Labels — Auto-Generated by Flan-T5')
    .hide(axis='index')
)
display(styled)
print(f'All {NUM_TOPICS} LDA topic labels generated!')

from gensim.models.coherencemodel import CoherenceModel

# Coherence for our 30-group model
coherence_model = CoherenceModel(
    model=lda_model, texts=df['tokens_bigram'].tolist(),
    dictionary=dictionary, coherence='c_v'
)
coherence_score = coherence_model.get_coherence()
print(f'📐 LDA Coherence Score (c_v) with 30 topics: {coherence_score:.4f}')
print('   (Higher is better; typically 0.4–0.7 is good)\n')

# Optional: compare coherence across different topic counts
print('⏳ Comparing coherence for different topic counts (this may take ~2 min)...')
topic_range = [10, 15, 20, 25, 30, 35, 40]
coherence_scores = []
for n in topic_range:
    m = LdaModel(corpus=corpus, id2word=dictionary, num_topics=n,
                 random_state=42, passes=3)  # reduced for memory efficiency
    cm = CoherenceModel(model=m, texts=df['tokens_bigram'].tolist(),
                        dictionary=dictionary, coherence='c_v')
    coherence_scores.append(cm.get_coherence())
    print(f'   Topics={n:3d} → coherence={coherence_scores[-1]:.4f}')
    del m, cm  # free memory after each iteration

# Plot coherence curve
plt.figure(figsize=(8, 4))
plt.plot(topic_range, coherence_scores, 'o-', color='#c94a2a', linewidth=2, markersize=8)
plt.axvline(30, color='#4a7c59', linestyle='--', linewidth=1.5, label='Our model (30 topics)')
plt.xlabel('Number of Topics')
plt.ylabel('Coherence Score (c_v)')
plt.title('Coherence Score vs Number of Topics', fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('coherence_curve.png', dpi=120, bbox_inches='tight')
plt.show()
best_n = topic_range[coherence_scores.index(max(coherence_scores))]
print(f'\n✅ Best topic count by coherence: {best_n} (score={max(coherence_scores):.4f})')

# Interactive pyLDAvis
pyLDAvis.enable_notebook()
vis = gensimvis.prepare(lda_model, corpus, dictionary, mds='mmds')
pyLDAvis.display(vis)

### 4B — NMF (Non-negative Matrix Factorization)

In [ ]:
tfidf = TfidfVectorizer(max_features=2000, min_df=3, max_df=0.7)
tfidf_matrix = tfidf.fit_transform(df['clean_text_bigram'])
feature_names = tfidf.get_feature_names_out()
nmf = NMF(n_components=NUM_TOPICS, random_state=42, max_iter=500)
nmf.fit(tfidf_matrix)

print('Generating NMF topic labels via Flan-T5...')
nmf_topic_labels = {}
nmf_rows = []
for idx, topic in enumerate(nmf.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-9:-1]]
    label = generate_topic_label(top_words)
    nmf_topic_labels[idx] = label
    nmf_rows.append({
        'Topic No.': f'Topic {idx+1}',
        'Generated Label': label,
        'Top Keywords': ', '.join(top_words[:7])
    })
    print(f'  Topic {idx+1:2d} -> {label}')

df_nmf_topics = pd.DataFrame(nmf_rows)
styled_nmf = (
    df_nmf_topics.style
    .set_properties(**{
        'text-align': 'left',
        'font-size': '13px',
        'padding': '6px 14px',
        'color': '#000000',
        'background-color': '#ffffff'
    })
    .set_table_styles([{
        'selector': 'th',
        'props': [('background-color','#1a5276'),('color','#ffffff'),
                  ('font-size','13px'),('padding','8px 14px'),('text-align','left')]
    },{
        'selector': 'tr:nth-child(even)',
        'props': [('background-color','#eaf4fb'),('color','#000000')]
    },{
        'selector': 'tr:nth-child(odd)',
        'props': [('background-color','#ffffff'),('color','#000000')]
    },{
        'selector': 'td',
        'props': [('color','#000000')]
    },{
        'selector': 'caption',
        'props': [('font-size','15px'),('font-weight','bold'),
                  ('padding','12px'),('color','#1a5276'),('text-align','left')]
    }])
    .set_caption('NMF Topic Labels — Auto-Generated by Flan-T5')
    .hide(axis='index')
)
display(styled_nmf)
print('NMF topic labels generated!')

# ── NMF Coherence Comparison ──
# Convert NMF topics to word lists for coherence scoring
print('\n⏳ Computing NMF coherence score...')
nmf_topics_words = []
for idx, topic in enumerate(nmf.components_):
    top_word_indices = topic.argsort()[:-11:-1]  # top 10 words
    nmf_topics_words.append([feature_names[i] for i in top_word_indices])

from gensim.models.coherencemodel import CoherenceModel as CM2
nmf_coherence_model = CM2(
    topics=nmf_topics_words,
    texts=df['tokens_bigram'].tolist(),
    dictionary=dictionary,
    coherence='c_v'
)
nmf_coherence_score = nmf_coherence_model.get_coherence()
print(f'📐 NMF Coherence Score (c_v) with {NUM_TOPICS} topics: {nmf_coherence_score:.4f}')

# ── Side-by-side comparison table ──
from IPython.display import display
comparison_data = {
    'Model': ['LDA', 'NMF'],
    'Algorithm': ['Latent Dirichlet Allocation', 'Non-negative Matrix Factorization'],
    'Topics': [NUM_TOPICS, NUM_TOPICS],
    'Coherence Score (c_v)': [round(coherence_score, 4), round(nmf_coherence_score, 4)],
    'Better Model': [
        '✅ Yes' if coherence_score >= nmf_coherence_score else '❌ No',
        '✅ Yes' if nmf_coherence_score > coherence_score else '❌ No'
    ]
}
df_comparison = pd.DataFrame(comparison_data)

def highlight_winner(row):
    if row['Better Model'] == '✅ Yes':
        return ['background-color: #c8f7c5; color: #1a5e1a; font-weight: bold'] * len(row)
    return ['background-color: #fde8e8; color: #8b0000'] * len(row)

styled_comparison = (
    df_comparison.style
    .apply(highlight_winner, axis=1)
    .set_properties(**{'text-align': 'left', 'font-size': '13px', 'padding': '8px 16px'})
    .set_table_styles([{
        'selector': 'th',
        'props': [('background-color','#1a252f'),('color','white'),
                  ('font-size','14px'),('padding','10px 16px'),('text-align','left')]
    },{
        'selector': 'caption',
        'props': [('font-size','16px'),('font-weight','bold'),
                  ('padding','12px'),('color','#1a252f'),('text-align','left')]
    }])
    .set_caption('📊 LDA vs NMF — Coherence Score Comparison')
    .hide(axis='index')
)
display(styled_comparison)

# ── Bar chart comparison ──
fig_cmp, ax_cmp = plt.subplots(figsize=(7, 4))
models = ['LDA', 'NMF']
scores = [coherence_score, nmf_coherence_score]
colors = ['#2a7fc9' if s == max(scores) else '#c94a2a' for s in scores]
bars = ax_cmp.bar(models, scores, color=colors, edgecolor='white', linewidth=1.5, width=0.4)
for bar, score in zip(bars, scores):
    ax_cmp.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{score:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=12)
ax_cmp.set_ylim(0, max(scores) * 1.2)
ax_cmp.set_ylabel('Coherence Score (c_v)', fontsize=11)
ax_cmp.set_title('LDA vs NMF — Coherence Score Comparison', fontweight='bold', fontsize=13)
ax_cmp.spines[['top','right']].set_visible(False)
winner = 'LDA' if coherence_score >= nmf_coherence_score else 'NMF'
ax_cmp.annotate(f'🏆 Winner: {winner}', xy=(0.5, 0.92), xycoords='axes fraction',
                ha='center', fontsize=12, color='#1a5e1a', fontweight='bold')
plt.tight_layout()
plt.savefig('lda_vs_nmf_coherence.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'\n✅ LDA coherence  : {coherence_score:.4f}')
print(f'✅ NMF coherence  : {nmf_coherence_score:.4f}')
print(f'🏆 Winner         : {winner} (higher coherence = more interpretable topics)')

### 4C — Dominant Topic + Zero-Shot Predicted Labels per Document
> Each document is assigned its highest-probability LDA topic. The topic label is **auto-generated** by `google/flan-t5-base` (text2text generation) — no manual keyword rules, no candidate labels.

In [ ]:
def get_dominant_topic(doc_bow):
    topics = lda_model.get_document_topics(doc_bow)
    return max(topics, key=lambda x: x[1]) if topics else (0, 0.0)

_results = [get_dominant_topic(bow) for bow in corpus]
df['dominant_topic_id'] = [r[0] for r in _results]
df['topic_prob']        = [r[1] for r in _results]
df['dominant_topic'] = df['dominant_topic_id'].apply(lambda x: f'Topic {int(x)+1}')
df['topic_label']    = df['dominant_topic_id'].apply(lambda x: topic_labels.get(int(x), 'Unknown'))
df['topic_prob']     = df['topic_prob'].round(3)

print('✅ Dominant topic + predicted label assigned to each document.\n')
print(df[['group','dominant_topic','topic_label','topic_prob']].head(15).to_string(index=False))

---
## 💬 Step 5 — Sentiment Analysis

In [ ]:
print('⏳ Running sentiment analysis...')

def sentiment(text):
    blob = TextBlob(text[:2000])  # use more text for better accuracy
    p = blob.sentiment.polarity
    return pd.Series({
        'polarity': round(p, 3),
        'subjectivity': round(blob.sentiment.subjectivity, 3),
        'sentiment': 'Positive' if p > 0.05 else ('Negative' if p < -0.05 else 'Neutral')
    })

df[['polarity','subjectivity','sentiment']] = df['text'].apply(sentiment)

print('✅ Sentiment analysis complete!\n')
print('Overall distribution:')
print(df['sentiment'].value_counts().to_string())
print(f'\nAverage polarity    : {df["polarity"].mean():.3f}')
print(f'Average subjectivity: {df["subjectivity"].mean():.3f}')

# Sentiment by category
print('Sentiment breakdown by category:\n')
cat_sentiment = df.groupby('group')['polarity'].agg(['mean','std','count']).round(3)
cat_sentiment.columns = ['Avg Polarity','Std Dev','Doc Count']
cat_sentiment['Overall'] = cat_sentiment['Avg Polarity'].apply(
    lambda x: '😊 Positive' if x > 0.05 else ('😞 Negative' if x < -0.05 else '😐 Neutral')
)
print(cat_sentiment.sort_values('Avg Polarity', ascending=False).to_string())

---
## ✍️ Step 6 — Extractive Summarization (per category)

In [ ]:
def extractive_summary(text, n=2):
    sentences = sent_tokenize(text)
    if len(sentences) <= n:
        return ' '.join(sentences)
    words = word_tokenize(text.lower())
    freq = {}
    for w in words:
        if w not in stop_words and w.isalpha() and len(w) > 3:
            freq[w] = freq.get(w, 0) + 1
    scores = {s: sum(freq.get(w, 0) for w in word_tokenize(s.lower())) for s in sentences}
    top_set = set(sorted(scores, key=scores.get, reverse=True)[:n])
    return ' '.join([s for s in sentences if s in top_set])

print('📝 Representative summary per category\n' + '='*65)
for cat in df['group'].unique():
    # Pick the highest-confidence doc in this category
    subset = df[df['group'] == cat].sort_values('topic_prob', ascending=False)
    best_doc = subset.iloc[0]['text']
    summary = extractive_summary(best_doc, n=2)
    print(f'\n🗞️  {cat}')
    print(f'   {summary[:220]}...')

---
## 📊 Step 7 — Visualization Dashboard

In [ ]:
fig = plt.figure(figsize=(20, 18))
fig.suptitle('Dynamic Topic Analysis — 20 Newsgroups | 30 Groups Dashboard',
             fontsize=20, fontweight='bold', y=1.01)
plt.rcParams.update({'font.family': 'serif', 'axes.spines.top': False, 'axes.spines.right': False})

PALETTE = ['#c94a2a','#c8933a','#4a7c59','#3d4f5c','#7b5ea7','#2a7fc9',
           '#c92a7a','#4a6e7c','#8a6a2a','#2ac98a']

# ── 1. Word Cloud per dataset ──
ax1 = fig.add_subplot(3, 3, 1)
all_words = ' '.join(df['clean_text_bigram'].tolist())
wc = WordCloud(width=600, height=350, background_color='#0e0d0c',
               colormap='YlOrRd', max_words=100).generate(all_words)
ax1.imshow(wc, interpolation='bilinear')
ax1.axis('off')
ax1.set_title('Overall Word Cloud', fontweight='bold')

# ── 2. Sentiment Distribution ──
ax2 = fig.add_subplot(3, 3, 2)
s_counts = df['sentiment'].value_counts()
color_map = {'Positive':'#4a7c59','Neutral':'#c8933a','Negative':'#c94a2a'}
bars = ax2.bar(s_counts.index, s_counts.values,
               color=[color_map.get(s,'#888') for s in s_counts.index],
               edgecolor='white', linewidth=1.5)
for bar in bars:
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
             str(int(bar.get_height())), ha='center', va='bottom', fontweight='bold', fontsize=9)
ax2.set_title('Sentiment Distribution', fontweight='bold')
ax2.set_ylabel('Documents')

# ── 3. Avg Polarity by Category ──
ax3 = fig.add_subplot(3, 3, 3)
cat_pol = df.groupby('group')['polarity'].mean().sort_values()
colors3 = ['#c94a2a' if v < 0 else '#4a7c59' for v in cat_pol.values]
ax3.barh(cat_pol.index, cat_pol.values, color=colors3, edgecolor='white')
ax3.axvline(0, color='#555', linewidth=1, linestyle='--')
ax3.set_title('Avg Polarity by Category', fontweight='bold')
ax3.set_xlabel('Polarity')
ax3.tick_params(axis='y', labelsize=8)

# ── 4. Topic Distribution (LDA) ──
ax4 = fig.add_subplot(3, 3, 4)
# Use topic labels for x-axis
topic_label_counts = df['topic_label'].value_counts()
topic_counts = df['dominant_topic'].value_counts().sort_index()
ax4.barh(topic_label_counts.index, topic_label_counts.values,
         color=PALETTE[:len(topic_label_counts)], edgecolor='white')
ax4.set_title('Topic Label Distribution', fontweight='bold')
ax4.set_xlabel('Document Count')
ax4.tick_params(axis='y', labelsize=7)

# ── 5. Topic vs Category Heatmap ──
ax5 = fig.add_subplot(3, 3, 5)
heat_data = pd.crosstab(df['group'], df['dominant_topic'])
sns.heatmap(heat_data, ax=ax5, cmap='YlOrRd', linewidths=.5,
            cbar_kws={'shrink': .7}, annot=True, fmt='d', annot_kws={'size': 7})
ax5.set_title('Category × Dominant Topic', fontweight='bold')
ax5.tick_params(axis='x', rotation=45, labelsize=7)
ax5.tick_params(axis='y', labelsize=7)
ax5.set_ylabel('')

# ── 6. Polarity vs Subjectivity scatter ──
ax6 = fig.add_subplot(3, 3, 6)
cats = df['group'].unique()
cat_colors = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(cats)}
for cat in cats:
    sub = df[df['group'] == cat]
    ax6.scatter(sub['polarity'], sub['subjectivity'],
                c=cat_colors[cat], alpha=0.4, s=15, label=cat)
ax6.axvline(0, color='#ccc', linestyle='--', linewidth=1)
ax6.axhline(0.5, color='#ccc', linestyle='--', linewidth=1)
ax6.set_xlabel('Polarity')
ax6.set_ylabel('Subjectivity')
ax6.set_title('Polarity vs Subjectivity', fontweight='bold')
ax6.legend(fontsize=5, loc='upper left', ncol=2)

# ── 7. Top words per category (top 3 cats) ──
top3_cats = df['group'].value_counts().index[:3]  # show top 3 category word clouds
for i, cat in enumerate(top3_cats):
    ax = fig.add_subplot(3, 3, 7 + i)
    cat_words = ' '.join(df[df['group'] == cat]['clean_text_bigram'].tolist())
    wc_cat = WordCloud(width=400, height=250,
                       background_color='#1a1a2e',
                       colormap='cool', max_words=50).generate(cat_words)
    ax.imshow(wc_cat, interpolation='bilinear')
    ax.axis('off')
    short_name = cat.split('.')[-1].replace('_', ' ').title()
    ax.set_title(f'Word Cloud: {short_name}', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('topic_analysis_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dashboard saved!')

---
## 📄 Step 8 — Final Report

In [ ]:
from datetime import datetime

print('=' * 68)
print('       DYNAMIC TOPIC ANALYSIS — 20 NEWSGROUPS REPORT')
print(f'       Generated : {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print('=' * 68)

print(f'\n📊 DATASET OVERVIEW')
print(f'   Total documents  : {len(df):,}')
print(f'   Categories       : {df["group"].nunique()}')
print(f'   Total tokens     : {df["tokens"].map(len).sum():,}')
print(f'   Vocabulary size  : {len(dictionary):,}')

print(f'\n🗂️  LDA TOPICS ({NUM_TOPICS} topics)')
for i, topic in lda_model.print_topics(num_words=7):
    words = [w.split('*')[1].replace('"','').strip() for w in topic.split('+')]
    print(f'   Topic {i+1:2d} → {", ".join(words)}')

print(f'\n   Coherence Score (c_v): {coherence_score:.4f}')

print(f'\n💬 SENTIMENT SUMMARY')
for label, cnt in df['sentiment'].value_counts().items():
    pct = round(cnt / len(df) * 100, 1)
    print(f'   {label:10s}: {cnt:4d} docs ({pct}%)')
print(f'   Avg polarity     : {df["polarity"].mean():.3f}')
print(f'   Avg subjectivity : {df["subjectivity"].mean():.3f}')

print(f'\n📰 CATEGORY SENTIMENT RANKING (most positive → most negative)')
for cat, row in cat_sentiment.sort_values('Avg Polarity', ascending=False).iterrows():
    short = cat.split('.')[-1]
    print(f'   {short:30s} polarity={row["Avg Polarity"]:+.3f}  ({row["Overall"]})')

print('\n' + '=' * 68)
print('✅ Full analysis complete!')
print('=' * 68)

---
## ⬇️ Step 9 — Download Results

In [ ]:
import zipfile, os
from google.colab import files

# Save results CSV
df[['group','dominant_topic','topic_label','topic_prob','sentiment','polarity','subjectivity']]\
  .to_csv('topic_analysis_results.csv', index=False)

# Save topic label tables
df_topics.to_csv('lda_topic_labels.csv', index=False)
df_nmf_topics.to_csv('nmf_topic_labels.csv', index=False)

# Bundle all outputs into a single zip
zip_name = 'DynamicTopicAnalysis_outputs.zip'
output_files = [
    'topic_analysis_dashboard.png',
    'coherence_curve.png',
    'lda_vs_nmf_coherence.png',
    'topic_analysis_results.csv',
    'lda_topic_labels.csv',
    'nmf_topic_labels.csv',
]
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in output_files:
        if os.path.exists(fname):
            zf.write(fname)
            print(f'  Added: {fname}')
        else:
            print(f'  Skipped (not found): {fname}')

files.download(zip_name)
print(f'\nDownloaded: {zip_name}')